In [43]:
### automatically refresh the buffer
%load_ext autoreload
%autoreload 2

### solve the auto-complete issue

%config Completer.use_jedi = False
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=FutureWarning)

### lvl 2 setups (systerm)
import os
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap,LinearSegmentedColormap,BoundaryNorm
import matplotlib.dates as mdates
import geopandas as gpd
from shapely.geometry import Point
from datetime import datetime
import h5py
import numpy as np
np.set_printoptions(suppress=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# ENA

In [44]:
ds_CC = xr.open_mfdataset("/data/ggong/CloudSat/2B-CLDCLASS-LIDAR.P1_R05/ENA_5x5/*.nc", combine="by_coords")
ds_CW = xr.open_mfdataset("/data/ggong/CloudSat/2B-CWC-RVOD.P1_R05/ENA_5x5/*.nc", combine="by_coords")
ds_EC = xr.open_mfdataset("/data/ggong/CloudSat/ECMWF-AUX.P1_R05/ENA_5x5/*.nc", combine="by_coords")


t_common = np.intersect1d(
    np.intersect1d(ds_CC.time.values, ds_CW.time.values),
    ds_EC.time.values
)

ds_CC = ds_CC.sel(time=t_common).sortby("time")
ds_CW = ds_CW.sel(time=t_common).sortby("time")
ds_EC = ds_EC.sel(time=t_common).sortby("time")


ds_CC_sel = ds_CC[
    ["CloudLayerBase", "CloudLayerTop",
     "CloudFraction",  "lat", "lon",]
]

ds_CW_sel = ds_CW[
    ["Height", "Cloud_Liq_Water_Path","Liq_Geom_Mean_Radius" ]
]

ds_EC_sel = ds_EC[["Temperature"]]


ds_merged = xr.merge([ds_CC_sel, ds_CW_sel, ds_EC_sel], join="inner")
ds_merged = ds_merged.reset_index("ec_height")
ds_merged = ds_merged.drop_vars("ec_height")

In [45]:
ds_merged.to_netcdf('/data/ggong/CloudSat/merge_nc/ENA_merge5.nc')

# SGP

In [46]:
ds_CC = xr.open_mfdataset("/data/ggong/CloudSat/2B-CLDCLASS-LIDAR.P1_R05/SGP_5x5/*.nc", combine="by_coords")
ds_CW = xr.open_mfdataset("/data/ggong/CloudSat/2B-CWC-RVOD.P1_R05/SGP_5x5/*.nc", combine="by_coords")
ds_EC = xr.open_mfdataset("/data/ggong/CloudSat/ECMWF-AUX.P1_R05/SGP_5x5/*.nc", combine="by_coords")


t_common = np.intersect1d(
    np.intersect1d(ds_CC.time.values, ds_CW.time.values),
    ds_EC.time.values
)

ds_CC = ds_CC.sel(time=t_common).sortby("time")
ds_CW = ds_CW.sel(time=t_common).sortby("time")
ds_EC = ds_EC.sel(time=t_common).sortby("time")


ds_CC_sel = ds_CC[
    ["CloudLayerBase", "CloudLayerTop",
     "CloudFraction", "lat", "lon"]
]

ds_CW_sel = ds_CW[
    ["Height", "Cloud_Liq_Water_Path", "Liq_Geom_Mean_Radius",]
]

ds_EC_sel = ds_EC[["Temperature"]]


ds_merged = xr.merge([ds_CC_sel, ds_CW_sel, ds_EC_sel], join="inner")
ds_merged = ds_merged.reset_index("ec_height")
ds_merged = ds_merged.drop_vars("ec_height")

In [47]:
ds_merged.to_netcdf('/data/ggong/CloudSat/merge_nc/SGP_merge5.nc')